# Donut Fine-Tuning for SurveyPlan AI
This notebook contains the complete pipeline for fine-tuning the `naver-clova-ix/donut-base` model to extract structured data from survey plans. It includes parsing geometry, tables, and titles using custom schemas.

## 1. Setup & Environment
Pull the latest changes from the GitHub repository and install required dependencies. We also mount Google Drive to access the dataset.

In [ ]:
import os
# Check if the repository already exists to avoid double cloning
if not os.path.exists('survey-plan-AI'):
    !git clone https://github.com/14jadon14/survey-plan-AI.git
else:
    !git -C survey-plan-AI pull origin main

%cd survey-plan-AI

!pip install -q transformers datasets pytorch-lightning sentencepiece torchvision accelerate editdistance

from google.colab import drive
drive.mount('/content/drive')

## 2. Load Dataset
We load the images and `metadata.jsonl` from the Google Drive directory. The dataset is split into training and validation sets.

In [ ]:
from datasets import load_dataset
from pathlib import Path
import json
import os

# Path where the metadata.jsonl and crops are stored
DATA_DIR = Path('/content/drive/MyDrive/SurveyPlan AI/runs/donut_tuning')
os.makedirs(DATA_DIR, exist_ok=True)

try:
    # Pre-process metadata.jsonl to remove missing images before loading
    metadata_path = DATA_DIR / 'metadata.jsonl'
    if metadata_path.exists():
        valid_entries = []
        missing_count = 0
        with open(metadata_path, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line)
                img_path = DATA_DIR / entry['file_name']
                if img_path.exists():
                    valid_entries.append(line)
                else:
                    missing_count += 1
                    print(f"Skipping missing file: {entry['file_name']}")
        
        if missing_count > 0:
            print(f"Removed {missing_count} missing entries from metadata.jsonl")
            with open(metadata_path, 'w', encoding='utf-8') as f:
                f.writelines(valid_entries)

    # Load dataset using HuggingFace's imagefolder script
    dataset = load_dataset("imagefolder", data_dir=str(DATA_DIR), split="train", download_mode="force_redownload")
    dataset = dataset.train_test_split(test_size=0.1)
    train_dataset = dataset['train']
    val_dataset = dataset['test']
    print(f"SUCCESS: Loaded {len(train_dataset)} train and {len(val_dataset)} validation samples.")
except Exception as e:
    print(f"ERROR LOADING DATASET: {e}")

## 2.5 Hyperparameters
All tunable hyper-parameters parameters are exposed here. Adjust these values depending on your GPU VRAM, target accuracy, and total dataset size without needing to edit the training logic below.

In [ ]:
# === TUNABLE HYPERPARAMETERS ===
MAX_EPOCHS = 30
BATCH_SIZE = 2
NUM_WORKERS = 2
LEARNING_RATE = 2e-5
MAX_LENGTH = 512
IMAGE_RESOLUTION = {"height": 1280, "width": 960}
# ===============================

## 3. Configure Processor & Tokenizer
Donut requires a high-resolution input for readable text (1280x960). We also add our custom schema tokens to the tokenizer's vocabulary to ensure the model recognizes our specific JSON keys.

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from tqdm import tqdm
import json

model_id = "/content/drive/MyDrive/SurveyPlan AI/Models/Donut_Finetuned"
processor = DonutProcessor.from_pretrained(model_id)
processor.image_processor.size = IMAGE_RESOLUTION

# Dynamically extract schema tokens from the dataset
task_start_tokens = set()
schema_tokens = set()

def extract_tokens(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            schema_tokens.add(f"<{k}>")
            extract_tokens(v)
    elif isinstance(obj, list):
        for item in obj:
            extract_tokens(item)

print("Analyzing dataset for tokens...")
for item in tqdm(train_dataset):
    try:
        gt = item['ground_truth']
        if isinstance(gt, str): gt = json.loads(gt)
        gt_parse = gt.get('gt_parse', gt)
        
        if isinstance(gt_parse, dict) and len(gt_parse) == 1:
            root_key = list(gt_parse.keys())[0]
            task_start_tokens.add(f"<s_{root_key}>")
            extract_tokens(gt_parse[root_key])
    except FileNotFoundError:
        # Skip missing files if they weren't caught by pre-processing
        continue
    except Exception as e:
        print(f"Warning: Skipping an item due to error: {e}")

task_start_tokens = sorted(list(task_start_tokens))
schema_tokens = sorted(list(schema_tokens))

new_tokens = task_start_tokens + schema_tokens
for token in schema_tokens:
    new_tokens.append(token.replace("<", "</"))

processor.tokenizer.add_tokens(new_tokens)
print(f"\nDynamically discovered {len(task_start_tokens)} task tokens and {len(schema_tokens)} schema tokens!")

## 4. Model Initialization
We load the base Donut model and adjust its token embeddings to account for the newly added schema tokens.

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(model_id)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Dynamically set the decoder start token ID to prevent hallucination stuttering
if task_start_tokens:
    general_token_id = processor.tokenizer.convert_tokens_to_ids(task_start_tokens[0])
else:
    general_token_id = processor.tokenizer.convert_tokens_to_ids("<s_general>")

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = general_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
print(f"Model successfully initialized and resized. Decoder Start Token ID: {general_token_id}")


## 5. Dataset Loader Definition
Here we define the PyTorch Dataset. A custom `json2token` helper recursively converts the JSON ground truth into the sequence string format Donut natively understands (e.g. `<s_general><text>...</text></s_general>`).

In [ ]:
from torch.utils.data import Dataset, DataLoader
import json
import torch

def json2token(obj, sort_json_key: bool = True):
    """
    Converts a JSON object (dict) into a token sequence for training.
    """
    if isinstance(obj, dict):
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"]
        output = ""
        keys = sorted(obj.keys()) if sort_json_key else obj.keys()
        for k in keys:
            output += f"<{k}>"
            output += json2token(obj[k], sort_json_key)
            output += f"</{k}>"
        return output
    elif isinstance(obj, list):
        return "".join([json2token(item, sort_json_key) for item in obj])
    else:
        return str(obj)

class DonutDataset(Dataset):
    def __init__(self, dataset, processor, max_length=MAX_LENGTH, split="train", task_prompt="<s_general>"):
        super().__init__()
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length
        self.split = split
        self.task_prompt = task_prompt

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        
        gt = item['ground_truth']
        if isinstance(gt, str): gt = json.loads(gt)
        gt_parse = gt.get('gt_parse', gt)
        
        # Automatically select prompt based on root key
        prompt = self.task_prompt
        if prompt is None:
            prompt = task_start_tokens[0] if task_start_tokens else "<s_general>"
        if isinstance(gt_parse, dict) and len(gt_parse) == 1:
            root_key = list(gt_parse.keys())[0]
            if f"<s_{root_key}>" in task_start_tokens:
                prompt = f"<s_{root_key}>"
                # Unwrap the redundant root key to simplify sequence generation
                gt_parse = gt_parse[root_key]
                
        sequence = prompt + json2token(gt_parse) + self.processor.tokenizer.eos_token
        
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            sequence, add_special_tokens=False, 
            max_length=self.max_length, padding="max_length", 
            truncation=True, return_tensors="pt"
        )["input_ids"].squeeze()
        
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

# Optimize Data Loading for final dataset: increase batch_size and num_workers as needed
train_ds = DonutDataset(train_dataset, processor)
val_ds = DonutDataset(val_dataset, processor, split="val")
# TWEAK THESE HYPERPARAMETERS FOR FASTER TRAINING
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

## 6. PyTorch Lightning Module
We wrap the model in a PyTorch Lightning module. This handles the training loop and implements a `validation_step` that computes the **Character Error Rate (CER)** metric, logging it to the progress bar. We aim to keep CER below 0.05.

In [ ]:
import pytorch_lightning as pl
import editdistance

class DonutModelPLModule(pl.LightningModule):
    def __init__(self, model, processor, learning_rate=LEARNING_RATE):
        super().__init__()
        self.model = model
        self.processor = processor
        self.learning_rate = learning_rate

    def training_step(self, batch, batch_idx):
        outputs = self.model(pixel_values=batch["pixel_values"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        labels = batch["labels"].clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id
        
        prompt_ids = labels[:, :1] 
        outputs = self.model.generate(
            pixel_values, 
            decoder_input_ids=prompt_ids,
            max_length=self.model.decoder.config.max_position_embeddings,
            pad_token_id=self.processor.tokenizer.pad_token_id,
            eos_token_id=self.processor.tokenizer.eos_token_id,
            use_cache=True, return_dict_in_generate=True,
        )
        
        preds = self.processor.tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)
        gts = self.processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        dists = []
        for pred, gt in zip(preds, gts):
            # Compute CER (Character Error Rate)
            dist = editdistance.eval(pred.strip(), gt.strip())
            cer = dist / max(len(gt.strip()), 1)
            dists.append(cer)
            
        avg_cer = sum(dists) / len(dists)
        self.log("val_cer", avg_cer, prog_bar=True)
        return avg_cer

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

pl_module = DonutModelPLModule(model, processor)


## 7. Start Training
Initialize the PyTorch Lightning Trainer and begin the fine-tuning process. We set `log_every_n_steps` low enough to see frequent updates in the console.

In [ ]:
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, 
    accelerator="gpu", 
    precision="16-mixed",
    log_every_n_steps=1
)

print("Training STARTING...")
# Force model back into train mode (in case inference was run earlier)
model.train()
for param in model.parameters():
    param.requires_grad = True

print("Model is now in TRAIN mode and weights are unfrozen.")
trainer.fit(pl_module, train_loader, val_loader)

print("\n=== TRAINING COMPLETE ===")
print("Ensure the 'val_cer' metric reached acceptable low bounds over the epochs.")

## 8. Export Model
Save the fine-tuned model weights and the tokenizer configurations physically back to Google Drive.

In [ ]:
EXPORT_DIR = '/content/drive/MyDrive/SurveyPlan AI/Models/Donut_Finetuned'
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f"Saving to {EXPORT_DIR}...")
model.save_pretrained(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)
print("Success!")

## 9. Inference & Evaluation Test
This cell evaluates the model's performance on exactly one sample from the dataset. It prints the actual JSON alongside the model's generated JSON. Finally, it outputs a strict set of **Statistics** measuring how successfully the model memorized or learned the prompt.

In [ ]:
import editdistance
from PIL import Image

def run_direct_inference(image, task_prompt=None):
    if task_prompt is None:
        task_prompt = task_start_tokens[0] if task_start_tokens else "<s_general>"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    
    pixel_values = processor(image.convert("RGB"), return_tensors="pt").pixel_values
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids
    
    # Greedy decoding is favored for strict OCR extraction
    outputs = model.generate(
        pixel_values.to(device),
        decoder_input_ids=decoder_input_ids.to(device),
        max_length=512,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    
    sequence = processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    
    try:
        json_res = processor.token2json(sequence)
    except Exception:
        json_res = {"raw_sequence": sequence}
        
    return sequence, json_res

print("Preparing dataset sample for Inference...")
# Use `train_dataset` or `val_dataset` based on how many images you have allocated
sample_idx = 0 
sample_item = train_dataset[sample_idx]
sample_img = sample_item['image']

gt_raw = sample_item['ground_truth']
if isinstance(gt_raw, str): gt_raw = json.loads(gt_raw)

# Intelligently extract the appropriate task_prompt from ground truth schema
task_prompt = task_start_tokens[0] if task_start_tokens else "<s_general>"
gt_parse = gt_raw.get('gt_parse', gt_raw)
if isinstance(gt_parse, dict) and len(gt_parse) == 1:
    root_key = list(gt_parse.keys())[0]
    if f"<s_{root_key}>" in task_start_tokens:
        task_prompt = f"<s_{root_key}>"

# Build Ground Truth text sequence strictly for comparison metrics
inner_gt = gt_parse.get(root_key, gt_parse) if 'root_key' in locals() else gt_parse
target_sequence = task_prompt + json2token(inner_gt)

pred_seq, pred_json = run_direct_inference(sample_img, task_prompt)

char_dist = editdistance.eval(pred_seq, target_sequence)
cer_percentage = (char_dist / max(len(target_sequence), 1)) * 100
exact_match_seq = (pred_seq == target_sequence)

def count_exact_matches(gt, pred):
    if not isinstance(gt, dict) or not isinstance(pred, dict): return 0, 1
    matches = 0
    total = 0
    for k, v in gt.items():
        if isinstance(v, dict):
            m, t = count_exact_matches(v, pred.get(k, {}))
            matches += m
            total += t
        elif isinstance(v, list):
            # Simplified list (table) checking
            total += len(v)
            if k in pred and isinstance(pred[k], list):
                for i in range(min(len(v), len(pred[k]))):
                    m, t = count_exact_matches(v[i], pred[k][i])
                    # Add 1 if the whole row dictionary matched perfectly
                    if m == t and t > 0: matches += 1
        else:
            total += 1
            if k in pred and str(pred[k]).strip() == str(v).strip():
                matches += 1
    return matches, total

# Unwrap gt_parse for accurate field comparison
try:
    gt_inner = gt_raw.get('gt_parse', gt_raw)
    # Donut model sometimes predicts an extra wrapper, sometimes just the direct dict
    # the processor output is usually just the dict
    field_matches, field_total = count_exact_matches(gt_inner, pred_json)
    field_acc = (field_matches / max(field_total, 1)) * 100
except Exception:
    field_acc = 0.0
    field_matches = 0
    field_total = 1

print("\n" + "="*50)
print("INFERENCE EVALUATION")
print("="*50)
print(f"Task Prompt: {task_prompt}\n")
print("--- EXPECTED (Ground Truth) ---")
print(json.dumps(gt_raw, indent=2))
print("\n--- PREDICTED ---")
print(json.dumps(pred_json, indent=2))

print("\n" + "="*50)
print("STATISTICS")
print("="*50)
print(f"Character Error Rate (CER): {cer_percentage:.2f}%")
print(f"Sequence Exact Match:       {'Yes ✅' if exact_match_seq else 'No ❌'}")
print(f"Field-Level Exact Match:    {field_matches}/{field_total} fields correct ({field_acc:.1f}%)")


## 10. Active Learning Auto-Labeler
Once you have used your YOLO pipeline to save *new* blank crops to your Drive (where `ground_truth` is `{"gt_parse": {"text": ""}}`), you can run this cell! It will load your newly fine-tuned Donut model, predict the correct JSON tags for the new images, and overwrite the blank rows in your `metadata.jsonl` with the AI's best guess. This is the **Data Flywheel**!

In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

# Ensure model is loaded
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

DATA_DIR = "/content/drive/MyDrive/SurveyPlan AI/runs/donut_tuning"
metadata_path = os.path.join(DATA_DIR, "metadata.jsonl")
backup_path = os.path.join(DATA_DIR, "metadata_backup.jsonl")

if not os.path.exists(metadata_path):
    print(f"File not found: {metadata_path}")
else:
    # Create a quick backup just in case
    !cp "{metadata_path}" "{backup_path}"
    
    # Read current metadata
    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    new_lines = []
    labeled_count = 0
    
    print("Scanning metadata for blank templates...")
    for line in tqdm(lines, desc="Auto-Labeling"):
        record = json.loads(line)
        gt = record.get("ground_truth", "")
        
        # Check if it's the blank template exported by YOLO
        if '{"gt_parse": {"text": ""}}' in gt or '"text": ""' in gt:
            img_path = os.path.join(DATA_DIR, record["file_name"])
            label = record.get("label", "general")
            
            if os.path.exists(img_path):
                # 1. Determine Prompt (Schema root)
                task_prompt = "<s_general>"
                if "curve_data" in label:
                    task_prompt = "<s_lot_geometry>"
                elif "parcel_info" in label or "lot_id" in label:
                    task_prompt = "<s_parcel_info>"
                elif "table" in label:
                    task_prompt = "<s_tabular_data>"
                elif "dist" in label or "az" in label or "rad" in label or "arc" in label:
                    task_prompt = "<s_lot_geometry>"
                    
                # 2. Run Inference
                try:
                    img = Image.open(img_path).convert("RGB")
                    pred_seq, pred_json = run_direct_inference(img, task_prompt)
                    
                    # 3. Format Prediction into Ground Truth string
                    # Assuming pred_json directly resembles {"general": {"text": "123"}}
                    new_gt = {"gt_parse": pred_json}
                    record["ground_truth"] = json.dumps(new_gt, ensure_ascii=False)
                    labeled_count += 1
                except Exception as e:
                    print(f"Failed to infer {img_path}: {e}")
                    
        new_lines.append(json.dumps(record, ensure_ascii=False) + "\n")
        
    # Write back the new annotated metadata
    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)
        
    print(f"\n[SUCCESS] Auto-Labeled {labeled_count} new images using the Donut Model!")
    print("Open Google Drive, review the JSON file to fix minor mistakes, and you are ready to train again!")
